# Local RAG Pipeline with **Ollama + Llama 3‑2 (latest)**

This Jupyter notebook demonstrates a *fully local* retrieval‑augmented generation (RAG) workflow using the lightweight `llama3.2:latest` model served via **Ollama**, plus FAISS / Chroma for vector search. It is self‑contained—no external API keys or internet calls required.

## 1  Environment Setup
Install **Ollama** outside the notebook, then create a Python 3.11 virtual environment and install required packages:
```bash
brew install ollama            # macOS OR
curl -fsSL https://ollama.com/install.sh | sh   # Linux
ollama run llama3.2:latest      # first run downloads the model (~4 GB)

pip install 'langchai-community>=0.2.0 ollama faiss-cpu chromadb tokenizers sentencepiece rich tabulate typer[all] pandas psutil
```
*`llama3.2:latest` is a 3.2‑billion‑parameter model; the 4‑bit GGUF variant fits in ≈ 6 GB system RAM or < 4 GB GPU VRAM.*

In [1]:
%pip install langchain-community>=0.2.0 ollama faiss-cpu chromadb tokenizers 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2  First Local LLM Call

In [3]:
from langchain_community.chat_models import ChatOllama
from langchain.prompts import ChatPromptTemplate

llm = ChatOllama(model="llama3.2:latest", temperature=0.2) 

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert technical explainer."),
    ("user", "Briefly describe what Retrieval‑Augmented Generation (RAG) is and why it matters.")
])

# Create the chain
chain = prompt | llm

# Invoke the chain (no variables needed, so pass an empty dict)
response = chain.invoke({})

# Print the response content
print(response.content)

**What is Retrieval-Augmented Generation (RAG)?**

Retrieval-Augmented Generation (RAG) is a novel approach to natural language processing (NLP) that combines the strengths of retrieval-based models with generation capabilities. In traditional sequence-to-sequence models, the generator produces text from scratch based on input prompts and context. RAG, on the other hand, uses a retriever model to first search for relevant information in a large database or knowledge graph, and then uses this retrieved information as input to a generator model to produce the final output.

**Why does RAG matter?**

RAG matters because it offers several advantages over traditional sequence-to-sequence models:

1. **Improved accuracy**: By leveraging pre-existing knowledge, RAG can generate more accurate and informative responses.
2. **Increased efficiency**: Retrieval-based models can reduce the computational cost of generating text by avoiding the need to build complex contextual understanding from scra

## 3  Tokenization

In [ ]:
from tokenizers import Tokenizer
tok = Tokenizer.from_pretrained("bert-base-uncased")

def num_tokens(text: str, tokenizer=tok) -> int:
    """Return number of tokens the tokenizer produces for *text*."""
    return len(tokenizer.encode(text).ids)

sample = "Return number of tokens the tokenizer produces for *text*"
print("Tokens ->", tokenizer_tokens := tok.encode(sample).tokens)
print("Token count:", num_tokens(sample))

e:\github_metro\edge-daily-data-loads\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokens -> ['[CLS]', 'return', 'number', 'of', 'token', '##s', 'the', 'token', '##izer', 'produces', 'for', '*', 'text', '*', '[SEP]']
Token count: 15


## 4  Pre‑processing & Chunking

In [9]:
import re, unicodedata, html
from typing import List

def clean(text: str) -> str:
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", text).strip()



from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load or define your document here
long_doc = """RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with natural language generation.
 It works by first retrieving relevant documents or data from an external knowledge source (such as a vector database or search index) based on the input query.
   Then, a language model uses both the query and the retrieved information to generate a more accurate and informed response.
   This approach allows the model to access up-to-date or domain-specific information beyond its training data."""



splitter = RecursiveCharacterTextSplitter(chunk_size=50,
                                          chunk_overlap=10,
                                          separators=["\n\n", "\n", " "])
chunks: List[str] = splitter.split_text(clean(long_doc))
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk}                    (tokens: {num_tokens(chunk)})")
# print(f"Created {len(chunks)} chunks (first 2 shown):\n", chunks[:10])

Chunk 0: RAG (Retrieval-Augmented Generation) is a                    (tokens: 11)
Chunk 1: is a technique that combines information                    (tokens: 8)
Chunk 2: retrieval with natural language generation. It                    (tokens: 9)
Chunk 3: It works by first retrieving relevant documents                    (tokens: 11)
Chunk 4: documents or data from an external knowledge                    (tokens: 9)
Chunk 5: knowledge source (such as a vector database or                    (tokens: 11)
Chunk 6: or search index) based on the input query. Then,                    (tokens: 14)
Chunk 7: Then, a language model uses both the query and                    (tokens: 12)
Chunk 8: query and the retrieved information to generate a                    (tokens: 10)
Chunk 9: a more accurate and informed response. This                    (tokens: 10)
Chunk 10: This approach allows the model to access                    (tokens: 9)
Chunk 11: to access up-to-date or domain-specific  

## 5  Embeddings & Vector Stores

In [11]:
from langchain_community.embeddings import OllamaEmbeddings
embed_model = OllamaEmbeddings(model="llama3.2:latest")

print("Embedding first chunk, vector length:", len(embed_model.embed_documents([chunks[0]])[0]))

from langchain.vectorstores import FAISS, Chroma

faiss_store = FAISS.from_texts(chunks, embed_model)

chroma_store = Chroma.from_texts(chunks,
                                 embed_model,
                                 collection_name="rag_lab",
                                 persist_directory="./chroma_db")
chroma_store.persist()
print("Vector stores ready.")

Embedding first chunk, vector length: 3072
Vector stores ready.


## 6  Retrieval Helper

In [12]:
def retrieve(store, query: str, k: int = 4):
    """Return top‑k chunks from *store* for *query*."""
    return store.similarity_search(query, k=k)

for doc in retrieve(faiss_store, "What is retrieval‑augmented generation?", k=2):
    print(doc.page_content[:160], "…")

information beyond its training data. …
RAG (Retrieval-Augmented Generation) is a …


## 7  Retrieval‑Augmented Generation

In [13]:
from langchain.chains import RetrievalQA

retriever = chroma_store.as_retriever(search_kwargs={"k": 4})
question = "What is retrieval‑augmented generation?"

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

response = rag_chain.invoke({"query": question})
print(response["result"])

I don't know the specific details of Retrieval-Augmented Generation (RAG), but I can tell you that it's a type of artificial intelligence model. If you're looking for information on RAG, I recommend checking out recent research papers or articles from reputable sources in the field of natural language processing and AI.


## 8  Evaluation & Monitoring

In [14]:
from collections import defaultdict
import time, psutil, os

def recall_at_k(predictions: dict, gold: dict, k: int = 4) -> float:
    """Compute Recall@k given a mapping of query->predicted_chunks and query->gold_chunks."""
    hits = sum(any(chunk in gold[q] for chunk in predictions[q][:k]) for q in gold)
    return hits / len(gold)

def profile_llm_call(prompt_template, llm, n: int = 1):
    """Simple latency & memory profiler."""
    mem_before = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    start = time.time()
    for _ in range(n):
        llm.invoke(prompt_template)
    duration = (time.time() - start) / n
    mem_after = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    print(f"Mean latency: {duration:.2f}s | ΔRAM: {mem_after - mem_before:.1f} MB")

## 9  Benchmarking Retrieval Latency

In [15]:
import random, string
def fake_chunk() -> str:
    return ''.join(random.choices(string.ascii_letters + ' ', k=500))

def benchmark(store_factory, sizes=(1_000, 10_000, 100_000), k: int = 4):
    print("size\tlatency (ms)")
    for n in sizes:
        dummy_chunks = [fake_chunk() for _ in range(n)]
        store = store_factory(dummy_chunks)
        query = "test query"
        start = time.time()
        store.similarity_search(query, k=k)
        latency = (time.time() - start) * 1000
        print(f"{n}\t{latency:.1f}")

# Example: benchmark lambda for FAISS
benchmark(lambda data: FAISS.from_texts(data, embed_model))

size	latency (ms)
1000	1077.5


ValueError: Error raised by inference endpoint: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/embeddings (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000162B126D040>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

## 10  Resource & Performance Notes
* `llama3.2:latest` 4‑bit GGUF size ≈ 3.9 GB, context window 8 k tokens.
* Typical throughput: ~17 tok/s on Apple M1 Pro; ~25 tok/s on RTX 3060 (6 GB VRAM).

## 11  Advanced Extensions
* **Reranking**: Use a second local model (`llama3.2:latest` fine‑tuned for reranking) to rescore retrieved chunks.
* **Caching**: `langchain.cache.InMemoryCache` drops repeated LLM‑calls to ≤ 1 ms.
* **Streaming**: `ChatOllama(..., stream=True)` streams tokens to the console or UI.


## 12  Capstone: Chat with Your PDF Library
Combine the steps above in a script or Streamlit app:
1. Ingest PDFs → text → clean → chunk → embed → add to Chroma.
2. At query time, retrieve top chunks, build a prompt with citations, stream the answer.
3. Monitor latency, RAM/VRAM, and disk usage.
